In [11]:
import segmentation_models_pytorch as smp

from sklearn.model_selection import train_test_split 

import os, time, numpy as np
from typing import Dict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split


# Constants

In [12]:
DEVICE  = torch.device(f"cuda:0" if torch.cuda.is_available() else "cpu")
#DEVICE  = "cpu"
print(DEVICE)


cuda:0


In [13]:
base_path = "../Dataset"

dataset = base_path + "/s2-utm-33N-18E-242N-2018"
train_geojson_path = base_path + "/br-18E-242N-crop-labels-train-2018.geojson"

folder = "/DS3"
subfolder1 = "/Temporal Data"

data_path = dataset+folder
temporal_data_path = dataset+folder+subfolder1

classification_model_path = "../Classification models/Spatial"
temporal_classification_model_path = "../Classification models/Temporal"


In [ ]:
# -----------------------
# Dataset (temporal)
# -----------------------
class TemporalSegDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray, T):
        """
        X: (N,T,4,H,W), Y: (N,H,W)
        """
        assert X.ndim == 5 and Y.ndim == 3, "X=(N,T,4,H,W), Y=(N,H,W)"
        assert X.shape[0] == Y.shape[0], "X, Y must share N"
        assert X.shape[1] == T, f"T mismatch: X has {X.shape[1]}, expected {T}"
        self.X = X
        self.Y = Y

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx]).float()   # (T,4,H,W)
        y = torch.from_numpy(self.Y[idx]).long()    # (H,W)
        return x, y

def make_loaders_temporal(trainX, trainY, T, batch, val_ratio=0.2, workers=4, seed=42):
    X = np.load(trainX)  # (N,T,4,H,W)
    Y = np.load(trainY)  # (N,H,W)
    ds = TemporalSegDataset(X, Y, T)
    n_val = int(len(ds)*val_ratio) 
    n_tr = len(ds)-n_val
    g = torch.Generator().manual_seed(seed)
    tr, va = random_split(ds, [n_tr, n_val], generator=g)
    tr_ld = DataLoader(tr, batch_size=batch, shuffle=True, pin_memory=True, num_workers=workers)
    va_ld = DataLoader(va, batch_size=batch, shuffle=False, pin_memory=True, num_workers=workers)
    return tr_ld, va_ld

def make_loader_test(testX, testY, T, batch, workers=4):
    X = np.load(testX) 
    Y = np.load(testY)
    ds = TemporalSegDataset(X, Y, T=T)
    return DataLoader(ds, batch_size=batch, shuffle=False, pin_memory=True, num_workers=workers)

# -----------------------
# Temporal models
# -----------------------
class EarlyFusionSeg(nn.Module):
    """(B,T,4,H,W) -> reshape to (B,4T,H,W) -> DeepLabV3+"""
    def __init__(self, num_classes: int, backbone: str, T: int):
        super().__init__()
        self.T = T
        self.net = smp.DeepLabV3Plus(
            encoder_name=backbone, encoder_weights=None,
            in_channels=4*T, classes=num_classes
        )
    def forward(self, x):  # (B,T,4,H,W)
        B,T,C,H,W = x.shape
        if T != self.T: raise ValueError(f"Expected T={self.T}, got {T}")
        x = x.reshape(B, T*C, H, W)      # (B,4T,H,W)
        return self.net(x)

class EarlyFusionUNet(nn.Module):
    """(B,T,4,H,W) → (B,4T,H,W) → U-Net"""
    def __init__(self, num_classes: int, backbone: str, T: int):
        super().__init__()
        self.T = T
        self.net = smp.Unet(
            encoder_name=backbone, encoder_weights=None,
            in_channels=4*T, classes=num_classes
        )
    def forward(self, x):  # x: (B,T,4,H,W)
        B,T,C,H,W = x.shape
        if T != self.T: raise ValueError(f"Expected T={self.T}, got {T}")
        x = x.reshape(B, T*C, H, W)     # (B,4T,H,W)
        return self.net(x)

# -----------------------
# Loss & metrics
# -----------------------
class DiceLossMulti(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__() 
        self.eps=eps
    def forward(self, logits, target):
        probs = torch.softmax(logits, dim=1)
        B,K,H,W = probs.shape
        tgt1h = torch.zeros_like(probs).scatter_(1, target.unsqueeze(1), 1)
        inter = (probs*tgt1h).sum((0,2,3))
        union = probs.sum((0,2,3)) + tgt1h.sum((0,2,3))
        dice = (2*inter + self.eps) / (union + self.eps)
        return 1 - dice.mean()

def loss_fn(logits, y, ce_weight=None):
    ce = torch.nn.CrossEntropyLoss(weight=ce_weight)(logits, y)
    dice = DiceLossMulti()(logits, y)
    return 0.5*ce + 0.5*dice

@torch.no_grad()
def confusion_matrix(preds, target, num_classes, ignore_index=None):
    if ignore_index is not None:
        mask = target != ignore_index
        preds = preds[mask] 
        target = target[mask]
    k = num_classes*target.view(-1) + preds.view(-1)
    return torch.bincount(k, minlength=num_classes**2).reshape(num_classes, num_classes)

@torch.no_grad()
def metrics_from_conf(conf: torch.Tensor) -> Dict[str,float]:
    tp = torch.diag(conf).float()
    gt = conf.sum(1).float() 
    pd = conf.sum(0).float()
    union = gt + pd - tp
    iou = torch.where(union>0, tp/union.clamp(min=1), torch.zeros_like(tp))
    miou = iou.mean().item()
    pixacc = (tp.sum()/conf.sum().clamp(min=1)).item()
    dice = torch.where((gt+pd)>0, 2*tp/(gt+pd).clamp(min=1), torch.zeros_like(tp))
    return {"mIoU": miou, "pixel_acc": pixacc, "mF1": dice.mean().item(),
            "IoU_per_class": iou.cpu().tolist(), "F1_per_class": dice.cpu().tolist(),
            "support_per_class": gt.cpu().tolist()}

# -----------------------
# Train / Eval
# -----------------------
def train_epoch(model, loader, opt, scaler, device, ce_weight=None):
    model.train()
    tot, tsum = 0, 0.0
    
    for x,y in loader:
        x = x.to(DEVICE).float()      # [B, T, 4, 64, 64]
        y = y.to(DEVICE).long()  
        
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(x)
            loss = loss_fn(logits, y, ce_weight)
        scaler.scale(loss).backward()
        scaler.step(opt) 
        scaler.update()
        bs = x.size(0) 
        tot += bs 
        tsum += loss.item()*bs
    return tsum/max(tot,1)

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    model.eval()
    conf = torch.zeros((num_classes,num_classes), device=device, dtype=torch.int64)
    tot, tsum = 0, 0.0
    
    for x,y in loader:
        x = x.to(DEVICE).float()      # [B, 4, 64, 64]
        y = y.to(DEVICE).long()  
        with torch.amp.autocast('cuda'):
            logits = model(x) 
            loss = loss_fn(logits, y)
        preds = logits.argmax(1)
        conf += confusion_matrix(preds, y, num_classes)
        bs = x.size(0) 
        tot += bs 
        tsum += loss.item()*bs
    mets = metrics_from_conf(conf) 
    mets["loss"] = tsum/max(tot,1)
    return mets, conf


In [15]:
class ImageLabelDataset(Dataset):
    def __init__(self, image_array, label_array):
        self.images = torch.tensor(image_array, dtype=torch.float32)  # (N, 4, 64, 64)
        self.labels = torch.tensor(label_array, dtype=torch.float32)  # (N, 1, 64, 64)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.labels[idx]



In [ ]:
# Data
final_gan_data = np.load((temporal_data_path+'/14_final_TGAN_data.npy'))
final_gan_labels = np.load((temporal_data_path+'/15_final_TGAN_labels.npy'))
doy_list = np.load((temporal_data_path+'/13_selected_dates_DOY_normalised.npy'))

test_patch_labels = np.load('../Dataset/s2-utm-33N-17E-243N-2019/7_filtered_labels_64X64.npy')
test_patch_data = np.load('../Dataset/s2-utm-33N-17E-243N-2019/8_filtered_data_64X64.npy')

labels_upadated = np.where(final_gan_labels == -1, 0, final_gan_labels)
train_data, _, train_labels, _ = train_test_split(final_gan_data, labels_upadated, test_size=0.2, random_state= 2) 

image_data = np.transpose(train_data, (0, 1, 4, 2, 3)) # reshape to (B,T,C,H,W) for early-fusion model

dataset = ImageLabelDataset(image_data, train_labels)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

test_labels_upadated = np.where(test_patch_labels == -1, 0, test_patch_labels)
test_data, validation_data, test_labels, validation_labels = train_test_split(test_patch_data, test_labels_upadated, test_size=0.5, random_state= 2) 


validation_image_data = np.transpose(validation_data, (0, 1, 4, 2, 3))

val_dataset = ImageLabelDataset(validation_image_data, validation_labels)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)


In [17]:
SEED =2
EPOCHS = 50
T = 27

encoder_name = 'resnet101'

# Model
# model = EarlyFusionSeg(10, encoder_name, T).to(DEVICE)
model = EarlyFusionUNet(num_classes=10, backbone=encoder_name, T=T).to(DEVICE)


# Optim & sched
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler()

# Train with early stop on val mIoU
best, bad, patience = -1.0, 0, 2
# os.makedirs(args.out, exist_ok=True)
# best_path = os.path.join(args.out, "best.pt")
t0 = time.time()
for ep in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, loader, opt, scaler, DEVICE)
    val_mets, _ = evaluate(model, val_loader, DEVICE, 10)
    sched.step()
    print(f"[{ep:03d}/{EPOCHS}] tr_loss={tr_loss:.4f} | "
            f"val_loss={val_mets['loss']:.4f} | val_mIoU={val_mets['mIoU']:.4f} "
            f"| val_acc={val_mets['pixel_acc']:.4f}")

    if val_mets["mIoU"] > best:
        best = val_mets["mIoU"] 
        bad = 0
        #torch.save({"model": model.state_dict(), "epoch": ep, "best": best}, best_path)
    else:
        bad += 1
        if bad >= patience:
            print("Early stopping.") 
            break

print(f"Finished in {(time.time()-t0)/60:.1f} min. Best val mIoU={best:.4f}")





C:\Users\Sai Suhaas\AppData\Local\Temp\ipykernel_28476\1820122178.py:15: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[001/50] tr_loss=0.9234 | val_loss=0.5084 | val_mIoU=0.7271 | val_acc=0.8245
[002/50] tr_loss=0.4614 | val_loss=0.3291 | val_mIoU=0.8060 | val_acc=0.8816
[003/50] tr_loss=0.3545 | val_loss=0.3084 | val_mIoU=0.8135 | val_acc=0.8863
[004/50] tr_loss=0.3228 | val_loss=0.2484 | val_mIoU=0.8456 | val_acc=0.9058
[005/50] tr_loss=0.2830 | val_loss=0.2277 | val_mIoU=0.8564 | val_acc=0.9134
[006/50] tr_loss=0.2643 | val_loss=0.2168 | val_mIoU=0.8679 | val_acc=0.9209
[007/50] tr_loss=0.2461 | val_loss=0.2071 | val_mIoU=0.8709 | val_acc=0.9225
[008/50] tr_loss=0.2223 | val_loss=0.2154 | val_mIoU=0.8653 | val_acc=0.9191
[009/50] tr_loss=0.2315 | val_loss=0.1911 | val_mIoU=0.8775 | val_acc=0.9268
[010/50] tr_loss=0.2178 | val_loss=0.1947 | val_mIoU=0.8752 | val_acc=0.9253
[011/50] tr_loss=0.1932 | val_loss=0.1799 | val_mIoU=0.8858 | val_acc=0.9316
[012/50] tr_loss=0.1855 | val_loss=0.1691 | val_mIoU=0.8943 | val_acc=0.9368
[013/50] tr_loss=0.1718 | val_loss=0.1696 | val_mIoU=0.8928 | val_acc=0.9364

In [ ]:
test_patch_labels = np.load('../Dataset/s2-utm-33N-17E-243N-2019/10_combined_labels.npy')
test_patch_data = np.load('../Dataset/s2-utm-33N-17E-243N-2019/9_combined_data.npy')

test_labels_upadated = np.where(test_patch_labels == -1, 0, test_patch_labels)
test_data, _, test_labels, _ = train_test_split(test_patch_data, test_labels_upadated, test_size=0.5, random_state= 2) 


test_image_data = np.transpose(test_data, (0, 1, 4, 2, 3))

test_dataset = ImageLabelDataset(test_image_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)


In [18]:
test_image_data = np.transpose(test_data, (0, 1, 4, 2, 3))

test_dataset = ImageLabelDataset(test_image_data, test_labels)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)

test_mets, test_conf = evaluate(model, test_loader, DEVICE, 10)
print(f"[TEST] loss={test_mets['loss']:.4f} | acc={test_mets['pixel_acc']:.4f} "
        f"| mIoU={test_mets['mIoU']:.4f} | mF1={test_mets['mF1']:.4f}")
print("IoU_per_class: " + ",".join(f"{x:.4f}" for x in test_mets["IoU_per_class"]) + "\n")
print("F1_per_class: "  + ",".join(f"{x:.4f}" for x in test_mets["F1_per_class"])  + "\n")


# np.save(os.path.join(args.out, "test_confusion.npy"), test_conf.cpu().numpy())
# with open(os.path.join(args.out, "test_metrics.txt"), "w") as f:
#     for k,v in test_mets.items():
#         if isinstance(v, (list,tuple)): continue
#         f.write(f"{k}: {v}\n")
#     f.write("IoU_per_class: " + ",".join(f"{x:.4f}" for x in test_mets["IoU_per_class"]) + "\n")
#     f.write("F1_per_class: "  + ",".join(f"{x:.4f}" for x in test_mets["F1_per_class"])  + "\n")


[TEST] loss=2.9105 | acc=0.3711 | mIoU=0.1510 | mF1=0.2315
IoU_per_class: 0.3128,0.3321,0.1851,0.0001,0.0132,0.3039,0.0003,0.0000,0.3612,0.0016

F1_per_class: 0.4766,0.4986,0.3124,0.0001,0.0261,0.4662,0.0006,0.0000,0.5308,0.0032

